# Exploring the FS-Mol benchmark

THEMAP's meta-learning results are reported against [FS-Mol](https://github.com/microsoft/FS-Mol),
a few-shot benchmark of ChEMBL bioactivity assays. This notebook is a guided tour of the data
itself — how many tasks there are, how big they are, which proteins they target, how chemically
varied they are, and how hard FS-Mol's own baselines find them.

The point is to build a mental model of the benchmark **before** looking at any THEMAP result.
Several of the things below are not obvious from the FS-Mol paper and change how the numbers
should be read.

---

## Before you start: you need the data

This notebook reads the FS-Mol **task files** — one gzipped JSON-lines file per assay.

> **Download:** [FS-Mol from FigShare](https://figshare.com/ndownloader/files/31345321) (~5 GB
> compressed). Unpack it so the repository contains:
>
> ```
> benchmarking_datasets/fsmol_datasets/
> ├── train/   *.jsonl.gz
> ├── valid/   *.jsonl.gz
> ├── test/    *.jsonl.gz
> ├── fsmol_tasks_list.json
> ├── fsmol_train_proteins.csv
> └── fsmol_test_proteins.csv
> ```

!!! warning "This is **not** the `make download-fsmol` archive"

`make download-fsmol` fetches a different 16 GB Zenodo archive — the companion data for the
THEMAP paper (precomputed OTDD matrices and ESM-2 embeddings), used by
the notebooks in `notebooks/paper/`. It does not contain the task files this notebook
needs, and this notebook does not need it.

**Runtime.** The first run scans every task file and takes about 4 minutes. Results are cached
under `notebooks/research/cache/` (gitignored), so later runs take seconds. Set `REBUILD_CACHE = True` in
the setup cell to force a rescan.

The setup cell locates the repository root on its own, so it does not matter which
directory you launch Jupyter from.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

# Resolve the repository root from anywhere inside the checkout, then work from there.
repo_path = next(str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
os.chdir(repo_path)
sys.path.insert(0, repo_path)

import gzip  # noqa: E402
import hashlib  # noqa: E402
import json  # noqa: E402
import re  # noqa: E402
import time  # noqa: E402
from collections import Counter  # noqa: E402

import matplotlib.colors as mcolors  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import matplotlib.ticker as mticker  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
from matplotlib.lines import Line2D  # noqa: E402
from matplotlib.patches import Patch  # noqa: E402
from scipy.stats import spearmanr  # noqa: E402
from sklearn.decomposition import PCA  # noqa: E402
from sklearn.manifold import TSNE  # noqa: E402
from sklearn.metrics.pairwise import cosine_distances  # noqa: E402

from themap.metalearning.fsmol_reference import (  # noqa: E402
    aggregate_reference,
    load_reference_table,
)
from themap.metalearning.subset import subset_representativeness, task_label_counts  # noqa: E402
from themap.utils import set_plot_style  # noqa: E402

DATA_DIR = Path("benchmarking_datasets/fsmol_datasets")
REFERENCE_DIR = Path("benchmarking_datasets/fsmol_reference")
SUBSET_FILE = Path("benchmarking_datasets/fsmol_subset_20.json")
CACHE = Path("notebooks/research/cache")
FOLD_ORDER = ["train", "valid", "test"]
REBUILD_CACHE = False

CACHE.mkdir(parents=True, exist_ok=True)

# Professional style: Set2 palette + CMU Serif font (Computer Modern look),
# with graceful fallback to the default serif font when CMU Serif is unavailable.
PAL = set_plot_style()
plt.rcParams["figure.figsize"] = (9, 5)

### Plot conventions

Two rules run through every figure here, and they are worth stating once.

**One colour per fold, everywhere.** `train` is teal, `valid` is orange, `test` is periwinkle,
from section 3 to section 11. Colour follows the entity, never its rank, so a reader who learns
the mapping once never has to re-read a legend.

**At most three categorical colours at a time.** Set2 is a pastel palette: past three slots its
hues stop being distinguishable under red-green colour blindness (`#e78ac3` and `#8da0cb` are
essentially the same colour to a protanope). So anything with more categories becomes small
multiples, an emphasis plot — one series in colour, the rest in grey — or a single-hue ramp when
the quantity is a magnitude rather than an identity.

In [ ]:
FOLD_COLOR = {"train": PAL[0], "valid": PAL[1], "test": PAL[2]}
GREY = "#b3b3b3"  # de-emphasis
INK = "#3a3a3a"  # annotation text


def hue_ramp(color, name="themap"):
    """A light-to-dark single-hue colormap, for encoding magnitude."""
    rgb = np.array(mcolors.to_rgb(color))
    return mcolors.LinearSegmentedColormap.from_list(
        name, [np.clip(rgb + (1.0 - rgb) * 0.82, 0, 1), rgb, rgb * 0.42]
    )


def ordinal_colors(color, n):
    """`n` steps of a single hue, none so light it disappears against white."""
    return [hue_ramp(color)(x) for x in np.linspace(0.3, 1.0, n)]


def tidy(ax, title=None, subtitle=None, xlabel=None, ylabel=None):
    """Title/subtitle pair over a recessive hairline grid."""
    if title:
        ax.set_title(title, loc="left", fontweight="bold", pad=20 if subtitle else 10)
    if subtitle:
        ax.text(0, 1.025, subtitle, transform=ax.transAxes, fontsize=10, color=INK, va="bottom")
    ax.set_xlabel(xlabel or "")
    ax.set_ylabel(ylabel or "")
    ax.grid(alpha=0.25, linewidth=0.6)
    ax.set_axisbelow(True)
    return ax


def swatches(ax, entries, marker="s", loc="lower right", **kwargs):
    """A frameless key built from colour proxies.

    CMU Serif has no filled-circle or filled-square glyph, so coloured bullets typed into a
    title come out as tofu. Always build the key from real handles.
    """
    handles = [
        Line2D([], [], marker=marker, linestyle="none", markersize=9, color=color, label=label)
        for label, color in entries
    ]
    return ax.legend(handles=handles, loc=loc, frameon=False, handletextpad=0.4, **kwargs)


def legend_below(ax, handles, ncol=4, y=-0.34, title=None):
    """Put a key under its own axes.

    Two charts side by side each need their own key: the same hue means "binding" in one and
    "exact" in the other, and one merged row would show teal twice with two meanings.
    """
    return ax.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(0.5, y),
        ncol=ncol,
        frameon=False,
        handletextpad=0.5,
        title=title,
    )


def label_ends(ax, entries, min_gap=0.075, x=1.015):
    """Direct-label series in a gutter to the right of the plot.

    Series that converge would otherwise stack their labels on top of one another, so anything
    closer than `min_gap` gets nudged up, and the whole column slides back down if it overshoots.
    """
    to_axes = ax.transAxes.inverted()
    ordered = sorted(entries, key=lambda item: item[2])
    placed = []
    for _, x_data, y_data, _ in ordered:
        _, fraction = to_axes.transform(ax.transData.transform((x_data, y_data)))
        if placed and fraction - placed[-1] < min_gap:
            fraction = placed[-1] + min_gap
        placed.append(fraction)
    overshoot = max(placed) - 1.0
    if overshoot > 0:
        placed = [fraction - overshoot for fraction in placed]
    for (label, _, _, color), fraction in zip(ordered, placed):
        ax.text(
            x,
            fraction,
            label,
            transform=ax.transAxes,
            color=color,
            fontsize=10.5,
            va="center",
            fontweight="bold",
        )


def log_ticks(ax, ticks):
    """Label a log x-axis at chosen sizes.

    Assay sizes span barely more than a decade, so matplotlib's default leaves a single
    "10^3" and no way to read anything off the axis.
    """
    ax.set_xticks(ticks, [f"{tick:,}" for tick in ticks])
    ax.xaxis.set_minor_locator(mticker.NullLocator())


def stacked_barh(ax, categories, series, colors, labels, min_label=7):
    """A 100% stacked horizontal bar with a hairline gap between segments."""
    y = np.arange(len(categories))
    left = np.zeros(len(categories))
    handles = []
    for values, color, label in zip(series, colors, labels):
        values = np.asarray(values, dtype=float)
        ax.barh(y, values, left=left, color=color, height=0.6, edgecolor="white", linewidth=1.4)
        handles.append(Patch(facecolor=color, label=label))
        for row, (value, start) in enumerate(zip(values, left)):
            if value >= min_label:
                ax.text(
                    start + value / 2,
                    row,
                    f"{value:.0f}%",
                    ha="center",
                    va="center",
                    fontsize=9.5,
                    color="white",
                )
        left += values
    ax.set_yticks(y, categories)
    ax.set_xlim(0, 100)
    ax.grid(axis="y", visible=False)
    return handles

### Does the data look right?

This cell fails loudly and early if the download is missing or incomplete, rather than letting
you get twenty cells in before something breaks.

In [ ]:
def preflight():
    """Check the FS-Mol layout, naming the download if anything is missing."""
    required = [DATA_DIR / "fsmol_tasks_list.json", DATA_DIR / "fsmol_test_proteins.csv"]
    required += [DATA_DIR / fold for fold in FOLD_ORDER]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError(
            "FS-Mol task files not found. Download them from\n"
            "  https://figshare.com/ndownloader/files/31345321\n"
            f"and unpack into {DATA_DIR}/.\n"
            "(This is NOT the `make download-fsmol` archive — see the notes at the top.)\n"
            "Missing: " + ", ".join(missing)
        )
    print(f"FS-Mol task files found under {DATA_DIR}")


preflight()

### The first trap: `train/` is not the training fold

`train/` holds **26,868** files, but the FS-Mol training fold is **4,938** tasks. The extras are
the unfiltered ChEMBL dump that ships in the same archive. Globbing the directory instead of
reading `fsmol_tasks_list.json` inflates every training statistic by more than 5×, so every scan
below goes through `load_split()`.

In [ ]:
def load_split():
    """The canonical 4938/40/157 fold membership."""
    with open(DATA_DIR / "fsmol_tasks_list.json") as handle:
        return json.load(handle)


split = load_split()
for fold in FOLD_ORDER:
    on_disk = len(list((DATA_DIR / fold).glob("*.jsonl.gz")))
    print(f"{fold:6s} {len(split[fold]):>6,} tasks in the fold   {on_disk:>6,} files on disk")

### Scanning the corpus

Each line of a task file is one molecule, carrying its SMILES, its binary label, the raw activity
value, the assay type — and also a 2048-bit ECFP fingerprint, 200 RDKit descriptors and a full
molecular graph. Those three account for almost the entire file.

That has two consequences worth exploiting. First, pulling individual fields out of the gzipped
bytes with a regex is roughly ten times cheaper than `json.loads` on every line — the same trick
`themap.metalearning.subset.task_label_counts` uses, which we reuse directly for label counts.
Second, **the fingerprints are already computed**, so every piece of chemistry below needs no
featurizer and no RDKit at all.

In [ ]:
_ASSAY_RE = re.compile(rb'"AssayType":\s*"([^"]*)"')
_RELATION_RE = re.compile(rb'"Relation":\s*"([^"]*)"')
_FP_RE = re.compile(rb'"fingerprints":\s*\[([^\]]*)\]')
_SMILES_RE = re.compile(rb'"SMILES":\s*"([^"]*)"')


def scan_assay_metadata(fold, task_ids):
    """Per-task assay type and activity-relation composition."""
    records = []
    for task_id in task_ids:
        types, relations = Counter(), Counter()
        with gzip.open(DATA_DIR / fold / f"{task_id}.jsonl.gz", "rb") as handle:
            for line in handle:
                match = _ASSAY_RE.search(line)
                if match:
                    types[match.group(1).decode()] += 1
                match = _RELATION_RE.search(line)
                if match:
                    relations[match.group(1).decode() or "missing"] += 1
        n = sum(types.values())
        records.append(
            {
                "task_id": task_id,
                "assay_type": types.most_common(1)[0][0] if types else "unknown",
                "assay_type_purity": (types.most_common(1)[0][1] / n) if n else np.nan,
                "frac_exact": relations.get("=", 0) / n if n else np.nan,
                "frac_censored": (relations.get(">", 0) + relations.get("<", 0)) / n if n else np.nan,
                "frac_relation_missing": relations.get("missing", 0) / n if n else np.nan,
            }
        )
    return pd.DataFrame.from_records(records)


def scan_molecule_sets(fold, task_ids):
    """Which molecules each task contains.

    Returns a per-task frame plus the fold's molecule set, so we can ask how much of one fold's
    chemistry also appears in another's. The `panel` hash groups tasks built from an identical
    compound list.
    """
    records, union = [], set()
    for task_id in task_ids:
        with gzip.open(DATA_DIR / fold / f"{task_id}.jsonl.gz", "rb") as handle:
            smiles = {m.group(1) for m in (_SMILES_RE.search(line) for line in handle) if m}
        union |= smiles
        records.append(
            {
                "task_id": task_id,
                "n_distinct": len(smiles),
                "panel": hashlib.md5(b"|".join(sorted(smiles))).hexdigest()[:12],
            }
        )
    return pd.DataFrame.from_records(records), union


def mean_pairwise_tanimoto(fingerprints, rng, max_molecules=400):
    """Mean Tanimoto over every molecule pair, subsampled for large assays."""
    if len(fingerprints) > max_molecules:
        fingerprints = fingerprints[rng.choice(len(fingerprints), max_molecules, replace=False)]
    if len(fingerprints) < 2:
        return np.nan
    intersection = fingerprints @ fingerprints.T
    popcount = fingerprints.sum(axis=1)
    union = popcount[:, None] + popcount[None, :] - intersection
    upper = np.triu_indices(len(fingerprints), k=1)
    numerator, denominator = intersection[upper], union[upper]
    valid = denominator > 0
    return float((numerator[valid] / denominator[valid]).mean())


def scan_fingerprints(fold, task_ids, seed=0):
    """Per-task ECFP centroid and internal chemical diversity."""
    rng = np.random.default_rng(seed)
    records, centroids = [], []
    for task_id in task_ids:
        rows = []
        with gzip.open(DATA_DIR / fold / f"{task_id}.jsonl.gz", "rb") as handle:
            for line in handle:
                match = _FP_RE.search(line)
                if match:
                    rows.append(np.array(match.group(1).split(b","), dtype=np.float32))
        fingerprints = (np.vstack(rows) > 0).astype(np.float32)
        centroids.append(fingerprints.mean(axis=0))
        records.append(
            {
                "task_id": task_id,
                "n_molecules": len(fingerprints),
                "mean_tanimoto": mean_pairwise_tanimoto(fingerprints, rng),
            }
        )
    return pd.DataFrame.from_records(records), np.vstack(centroids)

In [ ]:
def cached_frame(name, build):
    """Read `CACHE/name` if it exists, otherwise build it and save it."""
    path = CACHE / name
    if path.exists() and not REBUILD_CACHE:
        return pd.read_csv(path)
    started = time.time()
    frame = build()
    frame.to_csv(path, index=False)
    print(f"  built {name} in {time.time() - started:.0f}s")
    return frame


def per_fold(scan):
    """Run a scan over all three folds and stack the results."""
    return pd.concat([scan(fold, split[fold]).assign(fold=fold) for fold in FOLD_ORDER], ignore_index=True)


counts = cached_frame(
    "fsmol_task_counts.csv",
    lambda: pd.concat(
        [task_label_counts(DATA_DIR, fold=f, task_ids=split[f]).assign(fold=f) for f in FOLD_ORDER],
        ignore_index=True,
    ),
)
assay_meta = cached_frame("fsmol_assay_meta.csv", lambda: per_fold(scan_assay_metadata))


def build_molecule_sets():
    frames, unions = [], {}
    for fold in FOLD_ORDER:
        frame, union = scan_molecule_sets(fold, split[fold])
        frames.append(frame.assign(fold=fold))
        unions[fold] = union
    overlap = {
        "distinct_per_fold": {fold: len(union) for fold, union in unions.items()},
        "train_test_shared": len(unions["train"] & unions["test"]),
        "train_valid_shared": len(unions["train"] & unions["valid"]),
    }
    with open(CACHE / "fsmol_molecule_overlap.json", "w") as handle:
        json.dump(overlap, handle, indent=2)
    return pd.concat(frames, ignore_index=True)


molecule_sets = cached_frame("fsmol_molecule_sets.csv", build_molecule_sets)
with open(CACHE / "fsmol_molecule_overlap.json") as handle:
    overlap = json.load(handle)


def build_fingerprints():
    frames, blocks, folds = [], [], []
    for fold in FOLD_ORDER:
        frame, block = scan_fingerprints(fold, split[fold])
        frames.append(frame.assign(fold=fold))
        blocks.append(block)
        folds.extend([fold] * len(frame))
    stacked = pd.concat(frames, ignore_index=True)
    np.savez_compressed(
        CACHE / "fsmol_centroids.npz",
        centroids=np.vstack(blocks).astype(np.float32),
        task_id=np.array(stacked["task_id"]),
        fold=np.array(folds),
    )
    return stacked


diversity = cached_frame("fsmol_diversity.csv", build_fingerprints)
centroid_file = np.load(CACHE / "fsmol_centroids.npz", allow_pickle=True)
centroids, centroid_fold, centroid_ids = (
    centroid_file["centroids"],
    centroid_file["fold"],
    centroid_file["task_id"],
)

by_fold = {fold: counts[counts.fold == fold] for fold in FOLD_ORDER}
train_proteins = pd.read_csv(DATA_DIR / "fsmol_train_proteins.csv")
test_proteins = pd.read_csv(DATA_DIR / "fsmol_test_proteins.csv")
print(f"\n{len(counts):,} tasks scanned; centroids {centroids.shape}")

## 2. What is in the benchmark

FS-Mol is 5,135 assays split into three folds. The split is by *assay*, and the folds are
deliberately unequal: thousands of small training tasks, a handful of larger ones held out.

In [ ]:
rows = [
    ("tasks", lambda f: f"{len(by_fold[f]):,}"),
    ("molecule records", lambda f: f"{by_fold[f]['n'].sum():,}"),
    ("distinct molecules", lambda f: f"{overlap['distinct_per_fold'][f]:,}"),
    ("median assay size", lambda f: f"{int(by_fold[f]['n'].median()):,}"),
]

fig, ax = plt.subplots(figsize=(9, 3.9))
ax.set_axis_off()
for col, fold in enumerate(FOLD_ORDER):
    x = 0.05 + col * 0.32
    ax.add_patch(plt.Rectangle((x, 0.95), 0.26, 0.032, color=FOLD_COLOR[fold], transform=ax.transAxes))
    ax.text(x, 0.855, fold, transform=ax.transAxes, fontsize=13, fontweight="bold", color=INK)
    for row, (label, value) in enumerate(rows):
        y = 0.64 - row * 0.185
        ax.text(x, y, value(fold), transform=ax.transAxes, fontsize=16.5, color=FOLD_COLOR[fold])
        ax.text(x, y - 0.072, label, transform=ax.transAxes, fontsize=9.5, color=INK)
fig.tight_layout()
plt.show()

Note the gap between *molecule records* and *distinct molecules* already: the test fold has 56,220
rows but only 27,518 distinct compounds. Section 7 comes back to that.

## 3. How big is an assay?

This is the number that governs everything else in a few-shot benchmark: if a task has 40
molecules, a 64-shot support set is simply not available.

In [ ]:
fig, ax = plt.subplots(figsize=(9.6, 4.8))
labels = []
for fold in FOLD_ORDER:
    sizes = np.sort(by_fold[fold]["n"].to_numpy())
    y = np.arange(1, len(sizes) + 1) / len(sizes)
    ax.step(sizes, y, where="post", color=FOLD_COLOR[fold], lw=2.2)
    labels.append((fold, sizes[-1], y[-1], FOLD_COLOR[fold]))

ax.set_xscale("log")
ax.set_xlim(28, 6000)
ax.set_ylim(0, 1.05)
log_ticks(ax, [32, 64, 128, 256, 512, 1024, 2048, 4096])
for value, note, y_note in [
    (46, "median train\n46 molecules", 0.04),
    (157, "median test\n157 molecules", 0.45),
]:
    ax.axvline(value, color=GREY, lw=1, ls=":")
    ax.text(value * 1.1, y_note, note, fontsize=9, color=INK, va="bottom")

tidy(
    ax,
    "Test assays are roughly three times the size of training assays",
    "Cumulative share of tasks at or below a given assay size",
    "molecules in the assay (log scale)",
    "share of tasks in the fold",
)
label_ends(ax, labels)
fig.tight_layout(rect=[0, 0, 0.94, 1])
plt.show()

In [ ]:
bins = np.logspace(np.log10(30), np.log10(5200), 44)
fig, axes = plt.subplots(3, 1, figsize=(9, 5.4), sharex=True)
for ax, fold in zip(axes, FOLD_ORDER):
    ax.hist(by_fold[fold]["n"], bins=bins, color=FOLD_COLOR[fold])
    ax.set_xscale("log")
    log_ticks(ax, [32, 64, 128, 256, 512, 1024, 2048, 4096])
    ax.text(
        0.995,
        0.82,
        f"{fold}   {len(by_fold[fold]):,} tasks",
        transform=ax.transAxes,
        ha="right",
        fontsize=10.5,
        color=INK,
        fontweight="bold",
    )
    ax.grid(alpha=0.25, linewidth=0.6)
    ax.set_axisbelow(True)
    ax.set_ylabel("tasks")
axes[0].set_title(
    "Every fold is right-skewed; valid and test are also truncated from below",
    loc="left",
    fontweight="bold",
)
axes[-1].set_xlabel("molecules in the assay (log scale)")
fig.tight_layout()
plt.show()

The training fold runs down to 32 molecules and piles up around 40; valid and test were curated to
a floor near 130. That spike at exactly 157 molecules in both held-out folds is not a coincidence —
section 7 explains it.

The practical consequence is the next chart. A support-set size is only usable on tasks with more
molecules than the support set, and that constraint bites quickly.

In [ ]:
support_sizes = [16, 32, 64, 128, 256]
fig, ax = plt.subplots(figsize=(9.6, 4.4))
labels = []
for fold in FOLD_ORDER:
    sizes = by_fold[fold]["n"].to_numpy()
    frac = [(sizes > k).mean() for k in support_sizes]
    emphasised = fold == "test"
    color = FOLD_COLOR[fold] if emphasised else GREY
    ax.plot(
        support_sizes,
        frac,
        marker="o",
        ms=8,
        lw=2.6 if emphasised else 1.6,
        color=color,
        zorder=3 if emphasised else 2,
    )
    labels.append((fold, support_sizes[-1], frac[-1], color))

n_large = int((by_fold["test"]["n"] > 256).sum())
ax.annotate(
    f"only {n_large} of 157 test tasks\nsurvive a 256-shot support set",
    xy=(256, n_large / 157),
    xytext=(60, 0.52),
    fontsize=10,
    color=INK,
    arrowprops=dict(arrowstyle="-", color=INK, lw=0.8, shrinkA=0, shrinkB=8),
)
ax.set_xscale("log", base=2)
ax.set_xticks(support_sizes, [str(k) for k in support_sizes])
ax.set_xlim(14, 300)
ax.set_ylim(0, 1.05)
tidy(
    ax,
    "Large support sizes leave very few tasks to average over",
    "Share of tasks with more molecules than the requested support size",
    "support-set size",
    "share of tasks still usable",
)
label_ends(ax, labels)
fig.tight_layout(rect=[0, 0, 0.93, 1])
plt.show()

**Why this matters.** A support-256 result is an average over 43 tasks, not 157 — a different and
much noisier population than the support-16 result it sits beside in a table. On the training
side it is worse: asking for 64-shot episodes discards about 86% of the training pool, which is
why `EpisodeSampler` treats the requested shot count as a *maximum* rather than a requirement.

## 4. Class balance

FS-Mol thresholds each assay at its own median activity, so tasks are balanced by construction.
Worth confirming, then setting aside.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
bins = np.linspace(0.28, 0.72, 45)
ax.hist(by_fold["test"]["frac_pos"], bins=bins, density=True, color=FOLD_COLOR["test"])
ax.hist(
    by_fold["train"]["frac_pos"],
    bins=bins,
    density=True,
    histtype="step",
    lw=2,
    color=FOLD_COLOR["train"],
)
ax.axvline(0.5, color=INK, lw=1.2, ls="--")
ax.text(0.505, ax.get_ylim()[1] * 0.9, "perfect balance", fontsize=9.5, color=INK)

n_skewed = int((by_fold["test"]["frac_pos"].sub(0.5).abs() > 0.1).sum())
tidy(
    ax,
    "Labels are balanced by construction, so accuracy is never the interesting axis",
    f"Positive-class fraction per task — only {n_skewed} test tasks sit more than 10 points off 50/50",
    "fraction of molecules labelled active",
    "density",
)
ax.legend(
    handles=[
        Patch(facecolor=FOLD_COLOR["test"], label="test"),
        Patch(facecolor="none", edgecolor=FOLD_COLOR["train"], linewidth=2, label="train"),
    ],
    loc="upper right",
    frameon=False,
)
fig.tight_layout()
plt.show()

This is why FS-Mol reports **ΔAUPRC** — AUPRC minus the positive rate — rather than accuracy or
raw AUPRC. With prevalence pinned near 0.5, accuracy would be uninformative, and the small
residual spread in prevalence is exactly what the Δ corrects for.

It also explains a protocol choice in `themap fsmol-benchmark`: support sets are drawn
*proportionally stratified* rather than forced to 50/50, because for 18 test tasks a forced
balance would misrepresent the task.

## 5. What kind of measurement is each task?

Not every assay measures the same thing. FS-Mol keeps ChEMBL's assay-type code and the activity
relation, both of which affect how much you should trust a label.

In [ ]:
meta_by_fold = {fold: assay_meta[assay_meta.fold == fold] for fold in FOLD_ORDER}
shown = FOLD_ORDER[::-1]  # barh draws bottom-up; reverse so train reads at the top

fig, axes = plt.subplots(1, 2, figsize=(12.5, 3.9))

assay_codes = [("B", "binding"), ("F", "functional"), ("A", "ADME")]
handles_left = stacked_barh(
    axes[0],
    shown,
    [[(meta_by_fold[f]["assay_type"] == code).mean() * 100 for f in shown] for code, _ in assay_codes],
    [PAL[0], PAL[1], PAL[2]],
    [label for _, label in assay_codes],
)
tidy(axes[0], "What was measured", "Assay type, per task", "% of tasks in the fold")
legend_below(axes[0], handles_left, ncol=3)

rel_cols = [
    ("frac_exact", "exact (=)"),
    ("frac_censored", "censored (< or >)"),
    ("frac_relation_missing", "not recorded"),
]
handles_right = stacked_barh(
    axes[1],
    shown,
    [[meta_by_fold[f][col].mean() * 100 for f in shown] for col, _ in rel_cols],
    [PAL[0], PAL[1], GREY],
    [label for _, label in rel_cols],
)
tidy(
    axes[1],
    "How precisely it was measured",
    "Activity relation, averaged over molecules",
    "% of molecules in the fold",
)
legend_below(axes[1], handles_right, ncol=3)

fig.tight_layout(rect=[0, 0.1, 1, 1])
plt.show()

mixed = int((assay_meta["assay_type_purity"] < 1).sum())
print(f"tasks mixing more than one assay type: {mixed} of {len(assay_meta):,}")

Two things to take from this.

The assay type is **constant within a task** — all 5,135 of them — so it is a genuine per-task
attribute, not a per-molecule one. But the mix shifts between folds: the test fold is 29%
functional assays against the training fold's 15%. Functional readouts (cell-based, enzymatic
turnover) are noisier and less directly comparable across assays than binding measurements, so the
held-out fold is slightly harder than the training fold in a way that has nothing to do with
chemistry.

About one molecule in eight carries a **censored** activity (`>` or `<`) — the true value lies
beyond the assay's detection range. FS-Mol binarises these the same way as exact measurements, so
a fraction of every label set is, strictly speaking, a lower or upper bound.

In [ ]:
levels = sorted(
    set(train_proteins["confidence"].dropna().astype(int))
    | set(test_proteins["confidence"].dropna().astype(int))
)
frames = {"test": test_proteins, "train": train_proteins}

fig, ax = plt.subplots(figsize=(9, 3.2))
handles = stacked_barh(
    ax,
    ["test", "train"],
    [[(frames[name]["confidence"] == level).mean() * 100 for name in ["test", "train"]] for level in levels],
    ordinal_colors(PAL[0], len(levels)),
    [str(level) for level in levels],
    min_label=9,
)
tidy(
    ax,
    "Every test task has a confidently assigned protein target",
    "ChEMBL target-confidence score — 9 means a single, directly assigned protein",
    "% of tasks in the fold",
)
legend_below(ax, handles, ncol=6, y=-0.42, title="ChEMBL target-confidence score")
fig.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

Confidence is an *ordered* scale, so it gets a single-hue ramp rather than six categorical
colours — darker means more confident.

Every test task scores 8 or 9, while about 11% of training tasks score 7 or below, meaning the
protein assignment is indirect or ambiguous. Anything that reasons over the protein side of a
training task should treat that tail with suspicion.

## 6. Which proteins are these? — the train-to-test shift

Each task targets one protein, annotated in `fsmol_{train,test}_proteins.csv` with an EC class, a
protein family and a UniProt accession. This is where the benchmark is least uniform.

Two bookkeeping decisions first: 38% of training tasks have no EC annotation at all, and a handful
carry tuple-valued multi-EC strings like `"('transferase', 'hydrolase')"`. Both get their own
bucket rather than being silently dropped.

In [ ]:
def normalise_ec(value):
    """Collapse FS-Mol's tuple-valued multi-EC strings and blanks into named buckets."""
    if not isinstance(value, str) or not value.strip():
        return "unannotated"
    return "multiple" if value.startswith("(") else value


for frame in (train_proteins, test_proteins):
    frame["ec_class"] = frame["EC_super_class_name"].map(normalise_ec)
    frame["family"] = frame["protein_family"].fillna("unannotated")
    frame["super_family"] = frame["protein_super_family"].fillna("unannotated")

annotated_train = train_proteins[train_proteins["ec_class"] != "unannotated"]
annotated_test = test_proteins[test_proteins["ec_class"] != "unannotated"]
share = (
    pd.DataFrame(
        {
            "train": annotated_train["ec_class"].value_counts(normalize=True) * 100,
            "test": annotated_test["ec_class"].value_counts(normalize=True) * 100,
        }
    )
    .fillna(0.0)
    .sort_values("test")
)

fig, ax = plt.subplots(figsize=(9, 4.6))
y = np.arange(len(share))
light, dark = hue_ramp(PAL[0])(0.35), hue_ramp(PAL[0])(0.95)
ax.hlines(y, share["train"], share["test"], color=GREY, lw=2.2, zorder=1)
ax.scatter(share["train"], y, s=110, color=light, edgecolor="white", lw=1.5, zorder=3)
ax.scatter(share["test"], y, s=110, color=dark, edgecolor="white", lw=1.5, zorder=3)

# Label only the rows where the gap is the story; the axis carries the rest.
for row, (_, values) in enumerate(share.iterrows()):
    if abs(values["test"] - values["train"]) < 8:
        continue
    low, high = sorted((values["train"], values["test"]))
    ax.text(low - 1.5, row, f"{low:.0f}%", va="center", ha="right", fontsize=9, color=INK)
    ax.text(high + 1.5, row, f"{high:.0f}%", va="center", ha="left", fontsize=9, color=INK)

ax.set_yticks(y, share.index)
ax.set_xlim(-4, 92)
tidy(
    ax,
    "The test fold is far more concentrated than the training fold",
    "Share of annotated tasks per EC super-class",
    "% of annotated tasks in the fold",
)
swatches(
    ax,
    [(f"train (n={len(annotated_train):,})", light), (f"test (n={len(annotated_test)})", dark)],
    marker="o",
    loc="lower right",
)
ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()

In [ ]:
families = test_proteins["family"].value_counts()
top = families.head(12)[::-1]
labels = ["other families"] + list(top.index)
values = [int(families.iloc[12:].sum())] + list(top.values)
n_kinase = int((test_proteins["super_family"] == "kinase").sum())

fig, ax = plt.subplots(figsize=(9, 5.2))
colors = [PAL[2] if name.startswith("kinase") else GREY for name in labels]
bars = ax.barh(np.arange(len(values)), values, color=colors, height=0.72)
for bar, value in zip(bars, values):
    ax.text(
        bar.get_width() + 0.4,
        bar.get_y() + bar.get_height() / 2,
        str(value),
        va="center",
        fontsize=9,
        color=INK,
    )
ax.set_yticks(np.arange(len(labels)), labels)
ax.set_xlim(0, max(values) * 1.14)
tidy(
    ax,
    "Two-thirds of the test fold is one protein super-family",
    f"157 test tasks by protein family — {n_kinase} of them ({n_kinase / 157:.0%}) are kinases",
    "number of test tasks",
)
swatches(ax, [("kinase subfamily", PAL[2]), ("everything else", GREY)], loc="lower right")
ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()

So the held-out fold is not a random sample of the training distribution. Transferases go from 49%
of annotated training tasks to 80% of test tasks, and kinases specifically from 39% to 64%.

That is a deliberate design choice — FS-Mol wanted a test set that reflects where medicinal
chemistry data actually concentrates — but it means a benchmark score is largely a statement about
kinase assays, and generalisation to the long tail of protein classes is barely measured.

In [ ]:
per_target = train_proteins["target_chembl_id"].value_counts()
binned = per_target.clip(upper=10).value_counts().sort_index()
labels = [str(value) for value in binned.index[:-1]] + ["10+"]

fig, ax = plt.subplots(figsize=(9, 4.2))
ax.bar(np.arange(len(binned)), binned.values, color=hue_ramp(PAL[0])(0.8), width=0.72)
for position, value in enumerate(binned.values):
    ax.text(position, value + 12, str(value), ha="center", fontsize=9, color=INK)
ax.set_xticks(np.arange(len(binned)), labels)
tidy(
    ax,
    f"The {len(train_proteins):,} training assays cover only "
    f"{train_proteins['target_chembl_id'].nunique():,} distinct protein targets",
    f"One target carries up to {per_target.max()} separate assays — tasks are not independent draws",
    "assays measured against the same target",
    "number of targets",
)
ax.grid(axis="x", visible=False)
fig.tight_layout()
plt.show()

In [ ]:
shared_targets = set(train_proteins["target_chembl_id"].dropna()) & set(
    test_proteins["target_chembl_id"].dropna()
)
shared_accessions = set(train_proteins["target_accession_id"].dropna()) & set(
    test_proteins["target_accession_id"].dropna()
)
unannotated = (train_proteins["ec_class"] == "unannotated").mean()

print(f"train targets shared with test (ChEMBL id): {len(shared_targets)}")
print(f"train targets shared with test (UniProt):   {len(shared_accessions)}")
print(f"  {sorted(shared_accessions)}")
print(f"training tasks with no EC annotation:       {unannotated:.0%}")

Two facts to carry forward.

**The split is close to clean at the target level.** Out of 157 test targets, exactly one ChEMBL
target id and eight UniProt accessions also appear in training. Eight is small but not zero, and
they are all kinases — worth knowing before claiming a result is fully held out.

**Training tasks are not independent draws.** 4,938 assays cover 1,508 targets, with one target
carrying 69 separate assays. Sampling training tasks uniformly therefore over-samples a few
well-studied proteins, and a "distance to the nearest source task" is often a distance to one of
several near-duplicates.

## 7. The folds are smaller than they look

Section 2 flagged a gap between molecule *records* and *distinct* molecules. It is worth pulling
on, because it turns out to be the most consequential structural feature of the benchmark.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.0))
rows = ["test", "valid"]
shared = {"test": overlap["train_test_shared"], "valid": overlap["train_valid_shared"]}
distinct = overlap["distinct_per_fold"]

handles = stacked_barh(
    ax,
    rows,
    [
        [shared[f] / distinct[f] * 100 for f in rows],
        [(1 - shared[f] / distinct[f]) * 100 for f in rows],
    ],
    [PAL[1], GREY],
    ["also appears in a training assay", "not seen during training"],
)
for row, fold in enumerate(rows):
    ax.text(101, row, f"{distinct[fold]:,} distinct molecules", va="center", fontsize=9.5, color=INK)

tidy(
    ax,
    "The folds are split by assay, not by molecule",
    f"{shared['test'] / distinct['test']:.0%} of the test fold's molecules also occur "
    "somewhere in the training fold",
    "% of the fold's distinct molecules",
)
ax.set_xlim(0, 145)
ax.set_xticks(range(0, 101, 20))
legend_below(ax, handles, ncol=2, y=-0.46)
fig.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

This is not a bug. FS-Mol is a benchmark about generalising to a **new assay**, not to new
chemistry, so molecules are free to recur across tasks. But it does mean a model has very often
seen a test molecule during meta-training, just with a label from a different protein. Any claim
that reads as "predicting activity for unseen compounds" is not what this benchmark measures.

The next chart is the part that surprised me.

In [ ]:
panel_rows, panel_stats = [], {}
for fold in FOLD_ORDER:
    sizes = molecule_sets[molecule_sets.fold == fold]["panel"].value_counts()
    largest = int(sizes.iloc[0])
    other_shared = int(sizes[sizes > 1].sum() - largest)
    unique = int(sizes[sizes == 1].sum())
    total = largest + other_shared + unique
    panel_rows.append([largest / total * 100, other_shared / total * 100, unique / total * 100])
    panel_stats[fold] = dict(largest=largest, distinct_sets=len(sizes))

shown = FOLD_ORDER[::-1]
values = np.array([panel_rows[FOLD_ORDER.index(fold)] for fold in shown])

fig, ax = plt.subplots(figsize=(9, 3.6))
handles = stacked_barh(
    ax,
    shown,
    [values[:, 0], values[:, 1], values[:, 2]],
    [hue_ramp(PAL[1])(0.95), hue_ramp(PAL[1])(0.45), GREY],
    ["the single largest shared panel", "other repeated panels", "own molecule set"],
)
ax.annotate(
    f"one 157-compound panel, screened against\n"
    f"{panel_stats['test']['largest']} different kinases in test and "
    f"{panel_stats['train']['largest']} more in train",
    xy=(23, shown.index("test") - 0.32),
    xytext=(34, shown.index("test") - 0.75),
    fontsize=10,
    color=INK,
    va="top",
    arrowprops=dict(arrowstyle="-", color=INK, lw=0.8, shrinkA=0, shrinkB=4),
)
ax.set_ylim(-1.75, len(shown) - 0.4)
tidy(
    ax,
    "Nearly half the test fold is one compound panel measured over and over",
    f"157 test tasks are built from only {panel_stats['test']['distinct_sets']} distinct molecule sets",
    "% of tasks in the fold",
)
legend_below(ax, handles, ncol=3, y=-0.42)
fig.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

In [ ]:
test_panels = molecule_sets[molecule_sets.fold == "test"]
train_panels = molecule_sets[molecule_sets.fold == "train"]
shared_panels = set(train_panels["panel"]) & set(test_panels["panel"])

summary = pd.DataFrame(
    [
        {
            "molecules": int(test_panels[test_panels.panel == panel]["n_distinct"].iloc[0]),
            "train tasks": int((train_panels["panel"] == panel).sum()),
            "test tasks": int((test_panels["panel"] == panel).sum()),
        }
        for panel in shared_panels
    ]
).sort_values("test tasks", ascending=False)

print("Molecule sets that appear in both folds:")
print(summary.to_string(index=False))
print(
    f"\ntest tasks whose exact molecule set also exists in train: "
    f"{int(test_panels['panel'].isin(shared_panels).sum())} of 157"
)

**One 157-compound panel is screened against 357 different kinases** — 284 training tasks and 73
test tasks. That single panel accounts for 47% of the test fold and is the entire explanation for
the spike at exactly 157 molecules in section 3.

Half the test fold (78 of 157 tasks) has an exact molecule-set twin in the training fold. For
those tasks the *only* thing distinguishing target from source is the protein; the chemistry is
byte-identical. That is worth holding onto, because it bounds what any purely chemical
task-distance can possibly do — see section 10.

## 8. How varied are the molecules inside one assay?

Task size says how *many* molecules there are; it says nothing about whether they are a hundred
near-copies from one lead series or a hundred unrelated scaffolds. Mean pairwise Tanimoto over the
shipped ECFP answers that — high means tight and repetitive, low means broad.

In [ ]:
fig, ax = plt.subplots(figsize=(9.6, 4.6))
labels = []
for fold in FOLD_ORDER:
    values = np.sort(diversity[diversity.fold == fold]["mean_tanimoto"].dropna().to_numpy())
    y = np.arange(1, len(values) + 1) / len(values)
    ax.step(values, y, where="post", color=FOLD_COLOR[fold], lw=2.2)
    labels.append((fold, values[-1], y[-1], FOLD_COLOR[fold]))

panel_tanimoto = float(diversity[diversity.fold == "test"]["mean_tanimoto"].mode().iloc[0])
ax.annotate(
    "the shared kinase panel — 73 test tasks\nwith byte-identical chemistry",
    xy=(panel_tanimoto, 0.30),
    xytext=(0.21, 0.13),
    fontsize=10,
    color=INK,
    arrowprops=dict(arrowstyle="-", color=INK, lw=0.8, shrinkA=0, shrinkB=4),
)
ax.set_xlim(0.05, 0.85)
ax.set_ylim(0, 1.05)
tidy(
    ax,
    "Training assays are chemically tighter than test assays",
    "Mean pairwise Tanimoto similarity within a task (ECFP, subsampled at 400 molecules)",
    "mean within-assay Tanimoto similarity",
    "share of tasks in the fold",
)
label_ends(ax, labels)
fig.tight_layout(rect=[0, 0, 0.94, 1])
plt.show()

In [ ]:
merged = diversity.merge(counts[["task_id", "fold", "n"]], on=["task_id", "fold"])
train, test = merged[merged.fold == "train"], merged[merged.fold == "test"]

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.scatter(train["n"], train["mean_tanimoto"], s=7, color=GREY, alpha=0.35, linewidths=0)
ax.scatter(test["n"], test["mean_tanimoto"], s=26, color=FOLD_COLOR["test"], alpha=0.85, linewidths=0)

edges = np.logspace(np.log10(32), np.log10(5000), 13)
centres = np.sqrt(edges[:-1] * edges[1:])
medians = [
    train[(train.n >= low) & (train.n < high)]["mean_tanimoto"].median()
    for low, high in zip(edges[:-1], edges[1:])
]
ax.plot(centres, medians, color=INK, lw=1.8, ls="--")
ax.text(centres[-1] * 1.15, medians[-1], "median\ntrain task", fontsize=9, color=INK, va="center")

ax.set_xscale("log")
ax.set_xlim(28, 11000)
log_ticks(ax, [32, 128, 512, 2048, 8192])
tidy(
    ax,
    "Bigger assays cover more chemical space",
    "Each point is one task; the dashed line is the median training task per size band",
    "molecules in the assay (log scale)",
    "mean within-assay Tanimoto similarity",
)
swatches(ax, [("train", GREY), ("test", FOLD_COLOR["test"])], loc="upper right")
fig.tight_layout()
plt.show()

The typical training task has a mean internal Tanimoto around 0.45 — that is a single lead series,
a set of close analogues from one paper. Test tasks sit near 0.12, roughly as diverse as a random
screening deck.

So training and evaluation are not just different in size, they are different in kind: the model
meta-trains on narrow structure-activity problems and is evaluated on broad ones. The relationship
is largely mechanical — a 40-molecule assay comes from one series, a 1,000-molecule assay cannot —
but the effect on what "few-shot transfer" means here is real.

## 9. How do the tasks sit relative to each other?

One mean fingerprint per task gives one point per task, and 5,135 points can be laid out together.
Two principal components carry under 7% of the variance in 2,048 bit-frequency dimensions, so PCA
alone makes a shapeless blob; reducing to 50 components (43% of variance) and letting t-SNE handle
the neighbourhood structure produces a map worth looking at. It takes a few seconds.

In [ ]:
reduced = PCA(n_components=50, random_state=0).fit_transform(centroids)
xy = TSNE(n_components=2, perplexity=40, init="pca", random_state=0).fit_transform(reduced)

is_train, is_test = centroid_fold == "train", centroid_fold == "test"
test_ids = centroid_ids[is_test]
ec = test_proteins.set_index("chembl_id").reindex(test_ids)["EC_super_class_name"].fillna("other")
top_classes = ["transferase", "hydrolase", "oxidoreductase"]
ec = ec.where(ec.isin(top_classes), "other")
class_colors = [PAL[0], PAL[1], PAL[2], INK]

fig, ax = plt.subplots(figsize=(9, 6.4))
ax.scatter(xy[is_train, 0], xy[is_train, 1], s=7, color=GREY, alpha=0.3, linewidths=0)
for name, color in zip(top_classes + ["other"], class_colors):
    mask = (ec == name).to_numpy()
    ax.scatter(
        xy[is_test][mask, 0],
        xy[is_test][mask, 1],
        s=46,
        color=color,
        alpha=0.9,
        linewidths=0.8,
        edgecolor="white",
    )

panel_hash = test_panels["panel"].value_counts().index[0]
panel_tasks = set(test_panels[test_panels.panel == panel_hash]["task_id"])
panel_xy = xy[is_test][np.array([task in panel_tasks for task in test_ids])].mean(axis=0)
ax.annotate(
    "the shared kinase panel:\n357 tasks, one point",
    xy=tuple(panel_xy),
    xytext=(panel_xy[0] - 22, panel_xy[1] - 13),
    fontsize=10,
    color=INK,
    ha="center",
    arrowprops=dict(arrowstyle="-", color=INK, lw=0.9, shrinkA=0, shrinkB=8),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="none", alpha=0.85),
)
tidy(
    ax,
    "Test tasks sit inside the training distribution, not beside it",
    f"t-SNE of per-task mean ECFP over all {len(xy):,} tasks — test tasks coloured by EC super-class",
    "t-SNE 1",
    "t-SNE 2",
)
ax.set_xticks([])
ax.set_yticks([])
swatches(
    ax,
    [("train task", GREY)] + list(zip(top_classes + ["other"], class_colors)),
    loc="upper right",
)
fig.tight_layout()
plt.show()

Test tasks are scattered through the training distribution rather than off to one side — there is
no chemical domain shift between the folds, only an assay and target shift. The tight rings of
grey points are groups of tasks with identical centroids, the repeated panels from section 7.

The question THEMAP actually asks is narrower: for a given target task, how far away is the
nearest training task?

In [ ]:
gaps = cosine_distances(centroids[is_test], centroids[is_train]).min(axis=1)
nearest = pd.DataFrame({"task_id": test_ids, "nearest_source_distance": gaps})
nearest.to_csv(CACHE / "fsmol_nearest_source.csv", index=False)
n_identical = int((gaps < 1e-9).sum())

# The distribution is a spike at zero plus a long tail, which a histogram renders as one
# enormous bar. An ECDF shows the spike and the tail on the same axes.
fig, ax = plt.subplots(figsize=(9, 4.6))
ordered = np.sort(gaps)
ax.step(
    ordered,
    np.arange(1, len(ordered) + 1) / len(ordered),
    where="post",
    color=FOLD_COLOR["test"],
    lw=2.4,
)
ax.annotate(
    f"{n_identical} of 157 test tasks have a training task\nwith exactly the same molecules",
    xy=(0, n_identical / 157),
    xytext=(0.12, 0.33),
    fontsize=10,
    color=INK,
    arrowprops=dict(arrowstyle="-", color=INK, lw=0.8, shrinkA=0, shrinkB=4),
)
for rank, (_, row) in enumerate(nearest.nlargest(3, "nearest_source_distance").iterrows()):
    ax.annotate(
        row["task_id"],
        xy=(row["nearest_source_distance"], 1.0),
        xytext=(0.30, 0.70 - rank * 0.075),
        fontsize=8.5,
        color=INK,
        ha="right",
        va="center",
        arrowprops=dict(arrowstyle="-", color=INK, lw=0.7, shrinkA=2, shrinkB=2),
    )
ax.text(0.135, 0.775, "most isolated test tasks", fontsize=9.5, color=INK)

ax.set_xlim(-0.02, 0.52)
ax.set_ylim(0, 1.04)
tidy(
    ax,
    "For half the test fold, chemical distance cannot pick a source task",
    "Cosine distance from each test task's mean ECFP to its nearest training task",
    "distance to the nearest training task",
    "share of test tasks at or below",
)
fig.tight_layout()
plt.show()

## 10. What do FS-Mol's own baselines say?

FS-Mol publishes per-task results for seven baselines, cached in this repo under
`benchmarking_datasets/fsmol_reference/`. Loading them is what makes the rest of this notebook
actionable: we can ask which properties of a task predict how hard it turns out to be.

In [ ]:
reference = load_reference_table(REFERENCE_DIR, offline=True)
at16 = (
    reference[(reference.method == "PN") & (reference.support_size == 16)][["task_id", "delta_auprc"]]
    .rename(columns={"delta_auprc": "difficulty"})
    .dropna()
)

fig, ax = plt.subplots(figsize=(9, 4.3))
ax.hist(at16["difficulty"], bins=30, color=FOLD_COLOR["test"])
mean = at16["difficulty"].mean()
ax.axvline(mean, color=INK, lw=1.4, ls="--")
ax.text(mean + 0.008, ax.get_ylim()[1] * 0.9, f"mean {mean:.3f}", fontsize=9.5, color=INK)
tidy(
    ax,
    "Task difficulty varies enormously, and the headline number hides it",
    "FS-Mol ProtoNet ΔAUPRC per test task at support size 16",
    "ΔAUPRC (AUPRC above the random baseline)",
    "number of test tasks",
)
fig.tight_layout()
plt.show()

The published ProtoNet number for support 16 is a single mean of 0.211, but the per-task values run
from 0.008 to 0.449. Some FS-Mol test tasks are essentially solved at 16 examples; others are
indistinguishable from guessing. Averages across this population move for reasons that have
nothing to do with a method being better.

Two footnotes on these numbers, both from `docs/user-guide/fsmol-benchmark.md`. ΔAUPRC recomputed
from the per-task CSVs sits uniformly 0.005 above the paper's Table 2 — a detail of how the paper
averaged the prevalence term — so compare recomputed against recomputed. And the error bars below
are across *tasks*, not across seeds.

In [ ]:
METHOD_LABEL = {
    "PN": "ProtoNet",
    "GNN-MAML": "GNN-MAML",
    "GNN-MT": "GNN multitask",
    "RF": "random forest",
    "kNN": "kNN",
    "MAT": "MAT",
    "GNN-ST": "GNN single-task",
}
summary = aggregate_reference(reference)
highlight = {"PN": PAL[0], "RF": PAL[1]}

fig, ax = plt.subplots(figsize=(9.6, 4.8))
labels = []
for method in summary["method"].unique():
    frame = summary[summary.method == method].sort_values("support_size")
    color = highlight.get(method, GREY)
    ax.plot(
        frame["support_size"],
        frame["delta_auprc_mean"],
        marker="o",
        ms=6,
        lw=2.6 if method in highlight else 1.4,
        color=color,
        zorder=3 if method in highlight else 2,
    )
    labels.append(
        (
            METHOD_LABEL[method],
            frame["support_size"].iloc[-1],
            frame["delta_auprc_mean"].iloc[-1],
            color,
        )
    )

ax.set_xscale("log", base=2)
ax.set_xticks([16, 32, 64, 128, 256], ["16", "32", "64", "128", "256"])
tidy(
    ax,
    "A random forest on the same fingerprints catches up as support grows",
    "FS-Mol's seven published baselines, mean ΔAUPRC over the 157 test tasks",
    "support-set size",
    "mean ΔAUPRC",
)
label_ends(ax, labels, min_gap=0.062)
fig.tight_layout(rect=[0, 0, 0.82, 1])
plt.show()

Seven series would exhaust the palette and bury the point, so only the two that matter are in
colour. ProtoNet leads everywhere, but a plain random forest on the same ECFP fingerprints closes
most of the gap by support 256.

That is why the random forest is the acceptance criterion in `themap fsmol-benchmark`: it consumes
the same features THEMAP's meta-learners do, so losing to it cannot be blamed on a weaker encoder
and points at the meta-learning itself.

In [ ]:
sized = at16.merge(by_fold["test"][["task_id", "n"]], on="task_id")
rho_size = spearmanr(sized["n"], sized["difficulty"])

fig, ax = plt.subplots(figsize=(9, 4.4))
ax.scatter(sized["n"], sized["difficulty"], s=30, color=FOLD_COLOR["test"], alpha=0.8, linewidths=0)
ax.set_xscale("log")
log_ticks(ax, [128, 256, 512, 1024, 2048])
tidy(
    ax,
    "Bigger test assays are harder, not easier",
    f"Spearman ρ = {rho_size.statistic:.2f}  (p = {rho_size.pvalue:.1g}, n = {len(sized)})",
    "molecules in the assay (log scale)",
    "ProtoNet ΔAUPRC at support 16",
)
fig.tight_layout()
plt.show()

Larger assays are *harder*, not easier — which is the opposite of the usual intuition that more
data helps. The reason is section 8: a big assay is a chemically broad one, so 16 support examples
cover proportionally less of it. Size and difficulty are entangled, which is exactly why
`themap fsmol-subset` stratifies on both rather than either alone.

In [ ]:
ec_lookup = test_proteins[["chembl_id", "EC_super_class_name"]].rename(
    columns={"chembl_id": "task_id", "EC_super_class_name": "ec_class"}
)
by_class = at16.merge(ec_lookup, on="task_id").dropna(subset=["ec_class"])
order = by_class.groupby("ec_class")["difficulty"].median().sort_values().index.tolist()

fig, ax = plt.subplots(figsize=(9, 4.4))
rng = np.random.default_rng(0)
for row, name in enumerate(order):
    values = by_class[by_class.ec_class == name]["difficulty"].to_numpy()
    ax.scatter(
        values,
        row + rng.uniform(-0.16, 0.16, len(values)),
        s=26,
        color=FOLD_COLOR["test"],
        alpha=0.75,
        linewidths=0,
    )
    ax.plot([np.median(values)] * 2, [row - 0.3, row + 0.3], color=INK, lw=2)
    ax.text(0.52, row, f"n = {len(values)}", fontsize=9, color=INK, va="center")

ax.set_yticks(range(len(order)), order)
ax.set_xlim(-0.03, 0.62)
tidy(
    ax,
    "Only two protein classes have enough tasks to compare at all",
    "ProtoNet ΔAUPRC at support 16, by EC super-class; the vertical rule is the median",
    "ProtoNet ΔAUPRC at support 16",
)
ax.grid(axis="y", visible=False)
fig.tight_layout()
plt.show()

Transferases (125 tasks) and hydrolases (20) are the only classes with enough tasks to say
anything. The remaining five classes have one or two tasks each, so their apparent difficulty is
noise. Any per-class breakdown of a benchmark result on FS-Mol is really a statement about
transferases plus five anecdotes.

Finally, the question this notebook has been building towards: does chemical distance to the
training pool predict how hard a task is?

In [ ]:
bridge = at16.merge(nearest, on="task_id")
identical = bridge[bridge["nearest_source_distance"] <= 1e-9]
distinct_chem = bridge[bridge["nearest_source_distance"] > 1e-9]
rho_all = spearmanr(bridge["nearest_source_distance"], bridge["difficulty"])
rho_far = spearmanr(distinct_chem["nearest_source_distance"], distinct_chem["difficulty"])

fig, ax = plt.subplots(figsize=(9, 4.6))
ax.scatter(
    identical["nearest_source_distance"],
    identical["difficulty"],
    s=30,
    color=GREY,
    alpha=0.8,
    linewidths=0,
)
ax.scatter(
    distinct_chem["nearest_source_distance"],
    distinct_chem["difficulty"],
    s=32,
    color=FOLD_COLOR["test"],
    alpha=0.85,
    linewidths=0,
)
tidy(
    ax,
    "Chemical distance alone does not explain which tasks are hard",
    f"Spearman ρ = {rho_all.statistic:.2f} over all 157 tasks, "
    f"{rho_far.statistic:.2f} over the {len(distinct_chem)} with a non-zero distance",
    "distance to the nearest training task",
    "ProtoNet ΔAUPRC at support 16",
)
swatches(
    ax,
    [("a training task has identical chemistry", GREY), ("some chemical distance", FOLD_COLOR["test"])],
    loc="upper right",
)
fig.tight_layout()
plt.show()

print(
    bridge.assign(identical=bridge["nearest_source_distance"] <= 1e-9)
    .groupby("identical")
    .agg(tasks=("task_id", "size"), mean_delta=("difficulty", "mean"))
    .round(3)
    .to_string()
)

Across all 157 tasks there is a real negative correlation (ρ = −0.23): further from the training
pool means harder. But restricted to the 81 tasks where the distance is not exactly zero, it
vanishes (ρ = 0.02, p = 0.87).

In other words the apparent signal is carried almost entirely by the *identical versus not*
distinction, and the identical group is confounded — those are the shared-panel tasks, which are
also uniformly 157 molecules and all kinases.

The honest conclusion is narrow but useful: **cosine distance between mean ECFP vectors is too
crude a task distance for this benchmark.** It is not a verdict on distance-guided source
selection — it is the motivation for the machinery THEMAP actually uses, which compares full
label-aware distributions with OTDD and adds a protein-space term. For that, see
`external_chemical_hardness.ipynb`, `external_protein_hardness.ipynb` and `task_hardness.ipynb`.

## 11. The 20-task benchmark subset

`themap fsmol-subset` picks 20 of the 157 test tasks so the parity benchmark can be iterated on
quickly. Given everything above — difficulty spread, entangled size, one dominant protein class —
it is worth checking the subset is not accidentally an easy or unrepresentative slice.

In [ ]:
subset_ids = json.loads(SUBSET_FILE.read_text())["task_ids"]
sized["in_subset"] = sized["task_id"].isin(subset_ids)
rest, chosen = sized[~sized.in_subset], sized[sized.in_subset]

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.scatter(rest["n"], rest["difficulty"], s=28, color=GREY, alpha=0.7, linewidths=0)
ax.scatter(
    chosen["n"],
    chosen["difficulty"],
    s=80,
    color=FOLD_COLOR["test"],
    linewidths=1.4,
    edgecolor="white",
    zorder=3,
)
ax.set_xscale("log")
log_ticks(ax, [128, 256, 512, 1024, 2048])
tidy(
    ax,
    "The 20-task subset spans the whole size-by-difficulty plane",
    "Every test task, with the tasks chosen by `themap fsmol-subset` highlighted",
    "molecules in the assay (log scale)",
    "ProtoNet ΔAUPRC at support 16",
)
swatches(
    ax,
    [("the other 137 test tasks", GREY), ("the 20 chosen tasks", FOLD_COLOR["test"])],
    loc="upper right",
)
fig.tight_layout()
plt.show()

representativeness = subset_representativeness(reference, subset_ids)
print("How close the subset's reference means are to the full benchmark (ProtoNet):")
print(representativeness[representativeness.method == "PN"].round(3).to_string(index=False))

The subset covers the full plane and its ProtoNet reference means land within 0.02–0.035 of the
157-task means. Because FS-Mol publishes results *per task*, the benchmark recomputes the
reference on exactly the tasks being evaluated, so the comparison stays fair regardless — a
well-chosen subset just keeps those recomputed means close to the familiar ones.

Note the last row: at support 256 the subset has only 7 usable tasks out of 20, against 43 of 157.
That is section 3 again, and it is why the 256-shot column of any benchmark report deserves the
least trust.

## 12. What to carry into the results

1. **The training fold is 4,938 tasks, not the 26,868 files in `train/`.** Always filter through
   `fsmol_tasks_list.json`.
2. **Training and test assays differ in kind, not just in size.** Training tasks have a median of
   46 molecules and an internal Tanimoto near 0.45 — single lead series. Test tasks have 157
   molecules at a Tanimoto near 0.12 — broad screening decks.
3. **Large support sizes evaluate a different population.** Support 256 averages over 43 test
   tasks, not 157, and discards ~86% of the training pool.
4. **The test fold is a kinase benchmark.** 64% kinases and 80% transferases, against 39% and 49%
   of the annotated training fold. Only transferases and hydrolases have enough tasks for a
   per-class claim.
5. **Nearly half the test fold is one 157-compound panel** screened against 73 kinases, with 284
   more training tasks on the same panel. Those tasks are not independent evidence.
6. **The split is by assay, not by molecule.** 43% of test molecules appear in a training assay,
   and half the test tasks have a training task with byte-identical chemistry.
7. **Labels are balanced by construction**, which is why the metric is ΔAUPRC and why support sets
   are stratified proportionally rather than forced to 50/50.
8. **Per-task difficulty ranges from 0.008 to 0.449 ΔAUPRC.** A single benchmark mean hides that
   entirely, and a random forest on the same fingerprints nearly catches ProtoNet at support 256.
9. **Naive chemical distance does not predict difficulty here.** Mean-ECFP cosine correlates with
   ΔAUPRC only through the identical-versus-not split, which motivates the distribution-level and
   protein-space distances THEMAP actually uses.

### Where to go next

- `docs/user-guide/fsmol-benchmark.md` — running the parity benchmark against these tasks
- `metalearning_reproduction.ipynb` — distance-guided source selection end to end
- `task_hardness.ipynb`, `external_chemical_hardness.ipynb`, `external_protein_hardness.ipynb` —
  the paper's hardness measures (these need the separate `make download-fsmol` archive)